<a href="https://colab.research.google.com/github/TheraMind-Project-Team/Psychologist-Project-AI/blob/main/textProcessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import time
import ast
import glob
import os

In [ ]:
input_files = glob.glob("/content/drive/MyDrive/drive_exract/*TRANSCRIPT.csv")

data_list = []

for file_path in input_files:
    temp_df = pd.read_csv(file_path, sep='\\t', on_bad_lines='skip')
    file_name = os.path.basename(file_path)
    temp_df['source_file'] = file_name
    data_list.append(temp_df)
    print(f"Read and add {file_name}")

df = pd.concat(data_list, ignore_index=True)

In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
df

In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tadder')
nltk.download('averaged_perceptron_tadder_eng')

In [ ]:
from nltk import word_tokenize
df["value"] = df["value"].astype(str)
df["value"] = df["value"].apply(word_tokenize)
df.head(10)

In [ ]:
from nltk.corpus import stopwords
print(stopwords.words('english'))
en_stopwords = stopwords.words('english')

In [ ]:
word_stopwords = ["mhm", "aww", "um", "[laughter]", "uh" ]

In [ ]:
processed_column = []
for value in df["value"]:
    result = []
    for token in value:
        if token not in en_stopwords and token not in word_stopwords:
            result.append(token)
    processed_column.append(result)

df['value'] = processed_column

In [ ]:
pd.set_option('display.max_rows', None)
df

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

processed_column = []

for tokens in df['value']:
    doc = nlp(" ".join(tokens))
    result = [token.lemma_ for token in doc]
    processed_column.append(result)

df['value'] = processed_column
if 'pos_tags' in df.columns:
    df = df.drop(columns=['pos_tags'])
df


In [ ]:
processed_column = []

for token_list in df['value']:
    filtered_list = []
    for token in token_list:
        if len(token) >= 3:
            filtered_list.append(token)
    processed_column.append(filtered_list)

df['value'] = processed_column
df = df[df['value'].apply(len) > 0]

df

In [ ]:
output_filename = "/content/drive/MyDrive/drive_exract/process_text.csv"
df.to_csv(output_filename, index=False, sep=',')
print(f"Successfully saved in {output_filename}.")
output_filename

In [ ]:
pd.read_csv(output_filename).shape


In [ ]:
# start_time = time.time()

processed_text_file = "/content/drive/MyDrive/drive_exract/process_text.csv"
scores_file = "/content/drive/MyDrive/full_test_split.csv"
final_output1_file = "/content/drive/MyDrive/testing_data_text.csv"

try:
    df_text = pd.read_csv(processed_text_file, sep=',')
    df_text['value'] = df_text['value'].apply(ast.literal_eval)
    df_text['value_str'] = df_text['value'].apply(lambda tokens: ' '.join(tokens))
    df_participant = df_text[df_text['speaker'] == 'Participant'].copy()
    df_participant['Participant_ID'] = df_participant['source_file'].str.split('_').str[0].astype(int)
    df_add_text = df_participant.groupby('Participant_ID')['value_str'].apply(
        lambda rows: ' '.join(rows)
    ).reset_index()
    df_add_text.rename(columns={'value_str': 'Full_Text'}, inplace=True)
    print("Text compilation is complete")

    df_scores = pd.read_csv(scores_file)
    df_scores['Participant_ID'] = df_scores['Participant_ID'].astype(int)
    df_scores_needed = df_scores[['Participant_ID', 'PHQ_Binary', 'PHQ_Score']]
    print(f"The points file has been read: {scores_file}")

    df_final_data = pd.merge(
        df_scores_needed,
        df_add_text,
        on='Participant_ID',
        how='inner'
    )
    print("The points file has been read")

    df_final_data.to_csv(final_output1_file, index=False, encoding='utf-8')

except FileNotFoundError as e:
    if str(e).endswith(scores_file):
        print(f"Error: The points file was not found {scores_file}")
        print("Please make sure this file is in the same folder.")
    elif str(e).endswith(processed_text_file):
        print(f"Error: No text file found {processed_text_file} ")
    else:
        print(f"File error: {e}")
except Exception as e:
    print(f"File error: {e}")

In [ ]:
pd.read_csv("/content/drive/MyDrive/testing_data_text.csv")

In [ ]:
pd.read_csv("/content/drive/MyDrive/process_text.csv").shape

In [ ]:
file_train = "/content/drive/MyDrive/training_data_text.csv"
file_dev = "/content/drive/MyDrive/deving_data_text.csv"
file_test = "/content/drive/MyDrive/testing_data_text.csv"

try:
    df_train = pd.read_csv(file_train)
    df_dev = pd.read_csv(file_dev)
    df_test = pd.read_csv(file_test)

    print(f"Downloaded Training: {len(df_train)} صف")
    print(f"Downloaded Dev: {len(df_dev)} صف")
    print(f"Downloaded Testing: {len(df_test)} صف")

    combined_df = pd.concat([df_train, df_dev, df_test], ignore_index=True)

    columns_to_drop = ['PHQ8_Score', 'PHQ8_Binary', 'PHQ_Binary', 'PHQ_Score']

    combined_df.drop(columns=columns_to_drop, inplace=True, errors='ignore')
    combined_df = combined_df[['Participant_ID', 'Full_Text']]
    combined_df.dropna(subset=['Full_Text'], inplace=True)

    print(f"Columns after deletion: {combined_df.columns.tolist()}")
    output_filename = "/content/drive/MyDrive/all_text_data.csv"
    combined_df.to_csv(output_filename, index=False, encoding='utf-8')

    print(f"\n Successfully saved to the file: {output_filename}")
    print(combined_df.head())

except FileNotFoundError as e:
    print(f" Error: File not found: {e}")

In [ ]:
import pandas as pd
import os

# 1. أسماء الملفات
file_train = "training_data_text (1).csv"
file_dev = "deving_data_text.csv"
file_test = "testing_data_text.csv"

print("--- بدء عملية توحيد ودمج البيانات ---")

try:
    # 2. قراءة الملفات
    df_train = pd.read_csv(file_train)
    df_dev = pd.read_csv(file_dev)
    df_test = pd.read_csv(file_test)

    print(f"أعمدة Train الأصلية: {df_train.columns.tolist()}")
    print(f"أعمدة Test الأصلية: {df_test.columns.tolist()}")

    # 3. توحيد أسماء الأعمدة (Standardization)
    # سنقوم بإعادة تسمية أعمدة ملف الاختبار لتطابق ملف التدريب
    # هذا يمنع ظهور قيم NaN عند الدمج
    rename_map = {
        'PHQ_Binary': 'PHQ8_Binary',
        'PHQ_Score': 'PHQ8_Score'
    }
    df_test.rename(columns=rename_map, inplace=True)

    print("تم توحيد أسماء الأعمدة في ملف الاختبار.")

    # 4. دمج الملفات (Concatenate)
    combined_df = pd.concat([df_train, df_dev, df_test], ignore_index=True)

    print(f"\nإجمالي الصفوف بعد الدمج: {len(combined_df)}")

    # 5. حذف الأعمدة غير المرغوب فيها (الآن لها اسم موحد)
    # نحذف فقط الأعمدة التي لا نريدها، ونحتفظ بـ Participant_ID و Full_Text
    columns_to_keep = ['Participant_ID', 'Full_Text']

    # طريقة آمنة: نختار فقط الأعمدة التي نريدها بدلاً من حذف ما لا نريده
    # هذا يحميك من أي أعمدة غريبة أخرى قد تكون موجودة
    final_df = combined_df[columns_to_keep].copy()

    # 6. تنظيف نهائي
    # إزالة أي صفوف فارغة في النص
    final_df.dropna(subset=['Full_Text'], inplace=True)
    # التأكد من أن رقم المشارك صحيح (int)
    final_df['Participant_ID'] = final_df['Participant_ID'].astype(int)

    print(f"الأعمدة النهائية: {final_df.columns.tolist()}")

    # 7. حفظ الملف
    output_filename = "all_text_data_merged.csv"
    final_df.to_csv(output_filename, index=False, encoding='utf-8')

    print(f"\n✅ تم الحفظ بنجاح: {output_filename}")
    print(f"عدد العينات النهائية: {len(final_df)}")
    print(final_df.head())

except FileNotFoundError as e:
    print(f"❌ خطأ: لم يتم العثور على الملف: {e}")
except KeyError as e:
    print(f"❌ خطأ في أسماء الأعمدة: {e}")

In [ ]:
import pandas as pd
import os
import glob

print("--- 🧹 الخطوة 1 (تصحيح): تجميع نصوص المرضى فقط (بدون الدكتور) ---")

# 1. تحديد مكان الداتا الأصلية (عدلي المسار ده حسب مكان ملفاتك)
# هذا المسار المفترض لو الداتا على الدرايف
base_path = "/content/drive/MyDrive/drive_exract"

# قائمة لتخزين البيانات النظيفة
clean_data = []

# 2. البحث عن كل ملفات الـ Transcripts
# النمط: أي ملف ينتهي بـ _TRANSCRIPT.csv
transcript_files = glob.glob(os.path.join(base_path, "*/*_TRANSCRIPT.csv"))
print(f"📂 تم العثور على {len(transcript_files)} ملف Transcript.")

if len(transcript_files) == 0:
    print("❌ لم يتم العثور على ملفات! تأكدي من مسار 'base_path' الصحيح.")
else:
    print("⏳ جاري المعالجة والتنظيف...")

    for file_path in transcript_files:
        try:
            # استخراج رقم المريض من اسم الملف (مثلاً 303_TRANSCRIPT.csv -> 303)
            file_name = os.path.basename(file_path)
            participant_id = int(file_name.split('_')[0])

            # قراءة ملف المحادثة (الفاصل tab عادة في هذه الداتا)
            df_t = pd.read_csv(file_path, sep='\t')

            # --- 🔍 الفلتر السحري (أهم خطوة) ---
            # نأخذ فقط الصفوف التي يكون المتحدث فيها 'Participant'
            # (ونتجاهل 'Ellie' أو 'Interviewer')
            participant_text = df_t[df_t['speaker'] == 'Participant']['value'].astype(str)

            # دمج الكلام كله في نص واحد
            full_transcript = " ".join(participant_text)

            # تخزين النتيجة
            clean_data.append({
                'Participant_ID': participant_id,
                'Full_Text': full_transcript
            })

        except Exception as e:
            print(f"⚠️ خطأ في معالجة ملف {file_name}: {e}")

    # 3. دمج مع الدرجات (Scores)
    # نحمل ملف الدرجات عشان نربط كل نص بالدرجة بتاعته
    try:
        df_labels = pd.read_csv('train_split_Depression_AVEC2017.csv') # تأكدي من اسم ملف الدرجات
        df_clean = pd.DataFrame(clean_data)

        # دمج
        df_final = pd.merge(df_clean, df_labels, left_on='Participant_ID', right_on='Participant_ID', how='inner')

        # حفظ الملف الجديد النظيف
        output_file = "CLEAN_TEXT_DATA.csv"
        df_final.to_csv(output_file, index=False)

        print(f"\n🎉 تم الانتهاء! الملف الجديد النظيف: {output_file}")
        print(f"📊 عدد المرضى: {len(df_final)}")
        print("💡 الآن يمكنك استخدام هذا الملف في استخراج الـ Features وأنتِ مطمئنة.")

    except FileNotFoundError:
        print("❌ ملف الدرجات (train_split...) غير موجود لدمجه.")